# Gold — `gold_transacoes_cartao`

## Bibliotecas

In [0]:
from pyspark.sql import functions as F

## Variáveis

In [0]:
# origem
silver_transacoes = "novarota.silver.transacoes"
silver_cartoes    = "novarota.silver.cartoes"
silver_contas     = "novarota.silver.contas"
silver_estornos   = "novarota.silver.estornos"
silver_controle   = "novarota.silver.controle_cargas"

# destino
tabela_gold     = "novarota.gold.gold_transacoes_cartao"
tabela_controle = "novarota.gold.controle_gold"
nome_tabela     = "gold_transacoes_cartao"   # como essa tabela aparece no controle

print(f"Tabela alvo: {tabela_gold}")

## Ultima carga

In [0]:
# cargas que a silver terminou. O estornos é o último notebook da silver (e é ele que marca as estornadas)
cargas_silver = [
    l.data_referencia
    for l in spark.table(silver_controle).filter("tabela = 'estornos'").select("data_referencia").distinct().collect()
]

# cargas que essa tabela da gold já processou
if spark.catalog.tableExists(tabela_controle):
    cargas_processadas = [
        l.data_referencia
        for l in spark.table(tabela_controle).filter(f"tabela = '{nome_tabela}'").select("data_referencia").distinct().collect()
    ]
else:
    cargas_processadas = []

cargas_pendentes = sorted(c for c in cargas_silver if c not in cargas_processadas)

print(f"Cargas já processadas: {sorted(cargas_processadas)}")
print(f"Cargas pendentes: {cargas_pendentes}")

if not cargas_pendentes:
    dbutils.notebook.exit(f"Nada pendente pra {nome_tabela}")

## Meses afetados

In [0]:
cargas_sql = ", ".join(f"'{c}'" for c in cargas_pendentes)

query_meses = f"""
with tocados as (
-- meses das transações que chegaram nessas cargas (inclusive as atrasadas)
select trunc(data_transacao, 'MM') as mes
from {silver_transacoes}
where data_referencia in ({cargas_sql})

union

-- meses das transações que receberam estorno nessas cargas
select trunc(t.data_transacao, 'MM') as mes
from {silver_estornos} e
join {silver_transacoes} t on e.id_transacao = t.id_transacao
where e.data_referencia in ({cargas_sql})
)
,
ultimo_mes as (select trunc(max(data_transacao), 'MM') as mes from {silver_transacoes})


select mes from tocados

union

-- o mês seguinte também, por causa da comparação com o mês anterior no resumo
select add_months(t.mes, 1) as mes
from tocados t, ultimo_mes u
where add_months(t.mes, 1) <= u.mes
"""

meses = sorted(str(l.mes) for l in spark.sql(query_meses).collect())
meses_sql = ", ".join(f"'{m}'" for m in meses)

print(f"Meses a recalcular: {meses}")

## Transformações

In [0]:
query = f"""
with transacoes as (select *, trunc(data_transacao, 'MM') as mes
from {silver_transacoes}
where trunc(data_transacao, 'MM') in ({meses_sql}))


select
    t.id_transacao,
    t.data_transacao,
    t.mes,
    t.valor,
    t.nome_estabelecimento,
    t.mcc,
    t.cidade_estabelecimento,
    t.fl_estornada,
    c.id_cartao,
    c.bandeira,
    c.tipo_cartao,
    ct.id_conta,
    ct.tipo_conta,
    ct.id_cliente

from transacoes t
join {silver_cartoes} c  on t.id_cartao = c.id_cartao
join {silver_contas}  ct on c.id_conta  = ct.id_conta
"""

df = spark.sql(query)

print(f"Linhas dos meses afetados: {df.count()}")

## Conferência

In [0]:
dups_no_lote = (
    df.groupBy("id_transacao").count()
    .filter(F.col("count") > 1)
    .count()
)
print(f"Transações duplicadas no lote (esperado 0): {dups_no_lote}")

df.groupBy("mes").count().orderBy("mes").display()

## Gravação por mês

In [0]:
if not spark.catalog.tableExists(tabela_gold):
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(tabela_gold)
    )
    print(f"Tabela criada (carga inicial): {tabela_gold} | {df.count()} linhas")
else:
    # substitui só os meses afetados, o resto da tabela não é tocado
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("replaceWhere", f"mes in ({meses_sql})")
        .saveAsTable(tabela_gold)
    )
    print(f"Meses regravados em {tabela_gold}: {meses}")

## Controle

In [0]:
# registra as cargas que essa tabela processou. Fica por último: só conta se a gravação deu certo.
# o delete antes evita linha repetida no log se a célula rodar de novo
controle = (
    spark.createDataFrame([(nome_tabela, c) for c in cargas_pendentes], "tabela string, data_referencia string")
    .withColumn("processado_em", F.current_timestamp())
)
if spark.catalog.tableExists(tabela_controle):
    spark.sql(f"delete from {tabela_controle} where tabela = '{nome_tabela}' and data_referencia in ({cargas_sql})")
controle.write.format("delta").mode("append").saveAsTable(tabela_controle)

controle.display()

## Conferência pós-escrita

In [0]:
print(f"Conferência da tabela: {tabela_gold}")

df_gold = spark.table(tabela_gold)
print(f"Total de linhas: {df_gold.count()}")

dups = (
    df_gold.groupBy("id_transacao").count()
    .filter(F.col("count") > 1)
    .count()
)
print(f"Transações duplicadas (esperado 0): {dups}")

df_gold.groupBy("mes").count().orderBy("mes").display()